# CLIFFGUARD — round 3

Use a T4 GPU, then Run all. Expected runtime is 2.5–3.5 hours: roughly 30–45
minutes per model for the HH-RLHF generation, 10–15 for each XSTest baseline,
and 10–20 for each grading pass, plus cold model downloads.

**What this tests.** Whether the HH-RLHF FP16-versus-4.5-bit result persists
when the generation window is widened from 48 tokens to 256, and what the
labelled XSTest arm looks like at 256 tokens at full precision. It *tests*
those things; it does not settle the safety question. Two small models, one
fixed HH corpus, greedy decoding and a single NF4 first-token judge cannot do
that, and a result that survives here survives only those conditions.

The XSTest half supplies baselines that do not exist yet. The published
labelled result — an empty harmful-compliance cell in all 21 cells — was
measured at 48 tokens, where 97.8–100% of harmful deflections run into the cap.
Whether that cell is empty because the models withheld or because the window
closed first is not currently decidable, and these baselines are the first half
of deciding it.

**What it deliberately skips.** 5.5 bits, because three points cannot validate
a slope fitted on five and six rungs, and 4.5 bits is where every headline
sits. GSM8K, because at n=200 the result already sits exactly on the detection
threshold of its own design, and a larger set would support a claim the paper
declines to make. AWQ and GPTQ, because one extra scheme cannot establish
quantizer invariance and their install path is fragile unattended.

Note on naming: `RTN_4B` is group-64 round-to-nearest quantize/dequantize with
dense FP16 inference — about 4.5 stored bits per weight. It is not a deployed
INT4 kernel, and nothing here measures one.

Caches and completed runs are checkpointed to Drive. Re-running after a
disconnect restores finished work rather than repeating it, and validates what
it restores: a directory is only treated as complete if its manifest matches
the model, schemes, prompt count, token budget, seed and corpus hash expected,
and every completions file parses to the right number of rows.

## Environment

The setup is deliberately the same as round 2. It mounts Drive before any model work, so completed schemes survive a Colab disconnect.

In [ ]:
import os, sys, json, time, pathlib, platform, subprocess

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/parnish007/CLIFFGUARD.git'
# Set to the commit this notebook was validated against; '' follows main.
REPO_COMMIT = 'PIN_ME'   # set by the maintainer before handing this over
REPO_DIR = pathlib.Path('/content/CLIFFGUARD') if IN_COLAB else pathlib.Path.cwd()
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/cliffguard')

if IN_COLAB:
    # Fatal, not a warning. Without Drive every cache and run directory lives
    # on disk Colab wipes at disconnect, so an unattended session that loses
    # its connection at hour two loses the whole session. Continuing without it
    # is not a degraded run, it is a run that cannot survive the thing most
    # likely to happen to it.
    from google.colab import drive as _drive
    _drive.mount('/content/drive')
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    # Pinned to the commit this notebook was written against. A clone of
    # whatever main happens to be would silently run different analysis code,
    # and --depth 1 cannot reach a specific commit, hence the full clone above.
    if REPO_COMMIT:
        subprocess.run(['git', 'checkout', '--quiet', REPO_COMMIT], check=True)
    head = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True,
                          text=True, check=True).stdout.strip()
    print(f'repo commit  : {head}')
    if REPO_COMMIT and not head.startswith(REPO_COMMIT):
        raise SystemExit(f'checked out {head}, expected {REPO_COMMIT}')

    # check=True: a missing bitsandbytes surfaces as a CUDA error inside the
    # judge two hours from now rather than here.
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                    'bitsandbytes', 'datasets', 'accelerate'], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
import torch, numpy as np, transformers
HAS_GPU = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if HAS_GPU else 'NONE'
VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2) if HAS_GPU else 0.0
print(f'repo         : {pathlib.Path.cwd()}')
print(f'python       : {platform.python_version()}')
print(f'torch        : {torch.__version__}')
print(f'transformers : {transformers.__version__}')
print(f'numpy        : {np.__version__}')
print(f'GPU          : {GPU_NAME}  ({VRAM_GB} GB)')
if hasattr(os, 'statvfs'):
    st = os.statvfs('.')
    print(f'free disk    : {st.f_bavail * st.f_frsize / 1e9:.1f} GB')
if not HAS_GPU:
    raise SystemExit('No GPU. Runtime → Change runtime type → T4 GPU, then rerun this cell.')
if tuple(int(p) for p in transformers.__version__.split('.')[:2]) < (4, 45):
    raise SystemExit(f'transformers {transformers.__version__} too old (need >= 4.45).\nRun: !pip -q install -U transformers, then restart the runtime.')


## Restore

Drive state first, so the preflight below validates what will actually be used.

In [ ]:
# Restore Drive state BEFORE validating it. On a fresh Colab clone `data/` does
# not exist -- it is gitignored -- so the corpora can only come from Drive or a
# rebuild, and the run directories from a previous session likewise. Validating
# first would fail on files that were about to arrive, or pass a check on files
# that were never going to.
import shutil

def _restore(src: pathlib.Path, dst: pathlib.Path, what: str) -> bool:
    if not src.exists():
        print(f'[drive] no {what} to restore')
        return False
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f'[drive] restored {what} -> {dst}')
    return True

_restore(DRIVE_ROOT / 'fold_a', pathlib.Path('data/folds/fold_a'), 'Fold A corpus')
_restore(DRIVE_ROOT / 'eval_suites', pathlib.Path('data/eval_suites'), 'eval suites')
_restore(DRIVE_ROOT / 'artifacts' / 'runs', pathlib.Path('artifacts/runs'),
         'previous run directories')

# Rebuild only what is still missing. Fold A comes from an UNPINNED HuggingFace
# revision, so a rebuild can silently produce a different corpus and break the
# pairing with the 48-token runs; preflight hashes it immediately afterwards,
# which is what turns that risk into a caught error rather than a wasted
# session. XSTest is a single file at a stable URL and is safer to fetch.
if not pathlib.Path('data/folds/fold_a/anthropic_hh_refused.jsonl').exists():
    print('[corpus] Fold A missing; rebuilding (preflight will verify the hash)')
    subprocess.run([sys.executable, 'scripts/download_fold_a.py', '--download'],
                   check=True)
if not pathlib.Path('data/eval_suites/xstest.jsonl').exists():
    print('[corpus] XSTest missing; fetching')
    subprocess.run([sys.executable, 'scripts/download_eval_suites.py',
                    '--download', '--suites', 'xstest'], check=False)

# Cache the corpora back, so the next session restores instead of re-fetching.
if DRIVE_ROOT.exists():
    for src, name in ((pathlib.Path('data/folds/fold_a'), 'fold_a'),
                      (pathlib.Path('data/eval_suites'), 'eval_suites')):
        if src.exists():
            shutil.copytree(src, DRIVE_ROOT / name, dirs_exist_ok=True)
    print('[drive] corpora cached')

## Preflight

This gate runs before any expensive model load. Do not start the measurements if it reports a failure.

In [ ]:
# --require-xstest because this notebook's second half needs the labelled
# corpus. Without the flag a missing file is only a `skip`, preflight passes,
# and the failure surfaces after the two expensive HH-RLHF steps have already
# spent their GPU time.
preflight = subprocess.run([sys.executable, 'scripts/preflight_round2.py',
                            '--require-xstest'])
if preflight.returncode != 0:
    raise SystemExit(
        'PREFLIGHT FAILED. Do not start round 3; repair the reported gate '
        'failure first. Every check above runs on CPU in seconds, so fixing '
        'it costs nothing compared with discovering it at hour three.')

In [ ]:
MODELS_LONG = [('qwen3b', 'Qwen/Qwen2.5-3B-Instruct'), ('phi35', 'microsoft/Phi-3.5-mini-instruct')]
MODELS_XSTEST = MODELS_LONG + [('smollm17b', 'HuggingFaceTB/SmolLM2-1.7B-Instruct')]
JUDGE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
N_LONG, N_XSTEST, LONG_TOKENS, SEED = 250, 150, 256, 0
JUDGE_COMPLETION_CHARS, TAXONOMY_MAX_LENGTH = 2000, 2560
# Five verified single-token options rather than label-word prefixes.
# Under Qwen2.5 the old mode compared ' REF', ' COM', ' DEF', ' UNC'
# -- three-character prefixes shared with common words -- against
# ' DISCLAIM', an entire word. The five logits were therefore not
# commensurable, and DISCLAIM, the rarest observed class, was the one
# scored differently from the other four.
SCORING = 'letter'

# Caches go straight to Drive when it is mounted. The old mirror-between-arms
# design left completed schemes on Colab's disposable disk until a whole model
# had finished, so a disconnect could lose hours of valid cache entries.
CACHE_ROOT = (DRIVE_ROOT / 'artifacts') if DRIVE_ROOT.exists() else pathlib.Path('artifacts')
BEHAV_CACHE = str(CACHE_ROOT / 'behavioural_cache')
RESULTS = {}
# Verified by preflight; repeated here so the resume path can reject a
# restored directory whose prompts are not the ones we are pairing against.
FOLD_A_SHA = '7da25bf88ee0409ce4900a12052e15849a2898ed01cfdcdfe6409bbfc11bd9b5'
XSTEST_SHA = '33874ac77bd574a74283cd024466f442e69da870fa1195fcde8a9107433f9ce4'
print(f'long HH-RLHF : {N_LONG} per class, {LONG_TOKENS} tokens, FP16 + RTN 4-bit')
print(f'XSTest       : {N_XSTEST} per class, {LONG_TOKENS} tokens, FP16 only')
print(f'caches       : {CACHE_ROOT}' + ('' if DRIVE_ROOT.exists() else '   (LOCAL -- a disconnect loses them)'))

def run_step(label, script, args, timeout=10800):
    '''Stream one script invocation; keep its tail and exit status.

    The timeout is a watchdog rather than proc.wait(timeout=...). Reading a
    child's stdout to EOF blocks for as long as it lives, so wait was reached
    only after exit and could never stop a hung step. A watchdog kill is kept
    separate because Linux reports it as -9, the same code as an OOM kill.
    '''
    import threading
    cmd = [sys.executable, f'scripts/{script}'] + args
    print(f'\n$ {" ".join(cmd)}', flush=True)
    started, lines = time.time(), []
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    timed_out = []
    def _kill():
        timed_out.append(True)
        print(f'\n[{label}] no exit after {timeout / 3600:.1f} h; killing', flush=True)
        proc.kill()
    watchdog = threading.Timer(timeout, _kill)
    watchdog.daemon = True
    watchdog.start()
    try:
        for line in proc.stdout:
            if 'Loading weights' in line or 'it/s]' in line or 's/prompt' in line:
                continue
            lines.append(line.rstrip())
            print(line.rstrip(), flush=True)
        proc.wait()
    except Exception as exc:
        proc.kill()
        proc.wait()
        lines.append(f'ABORTED: {type(exc).__name__}: {exc}')
    finally:
        watchdog.cancel()
    ok = proc.returncode == 0
    RESULTS[label] = {'returncode': proc.returncode, 'timed_out': bool(timed_out), 'minutes': (time.time() - started) / 60, 'tail': lines[-40:]}
    print(f'\n=== {label}: {"OK" if ok else f"FAILED rc={proc.returncode}"} in {RESULTS[label]["minutes"]:.1f} min ===', flush=True)
    return ok

def run_step_resumable(label, script, args, attempts=3, timeout=10800):
    '''Retry only a real OOM. Each completed scheme is cached immediately, so a
    fresh process resumes farther through the ladder instead of starting over.
    A bad flag, missing checkpoint, or timeout cannot be improved by retrying.'''
    tag = label
    for attempt in range(1, attempts + 1):
        tag = label if attempt == 1 else f'{label}-retry{attempt}'
        if attempt > 1:
            print(f'\n[retry {attempt}/{attempts}] {label}: resuming from cache', flush=True)
        if run_step(tag, script, args, timeout=timeout):
            RESULTS[label] = RESULTS[tag]
            return True
        result = RESULTS[tag]
        if result['returncode'] != -9 or result['timed_out']:
            reason = 'timed out' if result['timed_out'] else f'rc={result["returncode"]}'
            print(f'[{label}] {reason} is not a resumable OOM; not retrying', flush=True)
            RESULTS[label] = result
            return False
    print(f'[{label}] still failing after {attempts} attempts', flush=True)
    RESULTS[label] = RESULTS[tag]
    return False

def free_vram():
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    # GPU allocation is released between schemes; host memory ratchets upward
    # across model loads and is what eventually triggers Colab's OOM killer.
    host = ''
    try:
        for line in pathlib.Path('/proc/meminfo').read_text().splitlines():
            if line.startswith('MemAvailable:'):
                host = f', host available {float(line.split()[1]) / 1e6:.1f} GB'
    except OSError:
        pass
    print(f'[vram] {torch.cuda.memory_allocated()/1e9:.2f} GB allocated{host}')

def checkpoint_to_drive():
    '''Mirror completed run directories to Drive; caches already live there.'''
    if not DRIVE_ROOT.exists():
        return
    import shutil
    for name in ('runs', 'behavioural_cache', 'sector_cache'):
        src = pathlib.Path('artifacts') / name
        if src.exists():
            shutil.copytree(src, DRIVE_ROOT / 'artifacts' / name, dirs_exist_ok=True)
    print(f'[drive] mirrored artifacts/ to {DRIVE_ROOT}')

def restore_from_drive():
    '''Bring back prior run directories before anything runs.'''
    if not DRIVE_ROOT.exists():
        return
    import shutil
    src = DRIVE_ROOT / 'artifacts' / 'runs'
    if src.exists():
        shutil.copytree(src, pathlib.Path('artifacts') / 'runs', dirs_exist_ok=True)
        print('[drive] restored artifacts/runs')


def latest_run(pattern):
    hits = sorted(pathlib.Path('artifacts/runs').glob(pattern))
    return hits[-1] if hits else None

def completed_run(label, schemes, model, n_prompts, corpus_sha=None,
                  tokens=None):
    '''A restored run directory that is genuinely finished, not merely present.

    Existence is not completion. A Drive copy interrupted mid-write, a stale
    directory from a run with different arguments, or a truncated JSON all look
    identical to `path.exists()`, and treating any of them as done would skip
    the step and hand the analysis silently wrong data. That is worse than
    re-running: a missing result is visible, a wrong one is not.

    So everything the step depends on is checked against the manifest --
    model, scheme list, prompt count, token budget, seed, and the ordered
    corpus hash that makes the comparison paired at all -- and every
    completions file is parsed and counted rather than stat-ed.
    '''
    tokens = LONG_TOKENS if tokens is None else tokens
    complete = []
    for run in sorted(pathlib.Path('artifacts/runs').glob(f'*_{label}')):
        try:
            manifest = json.loads((run / 'manifest.json').read_text(encoding='utf-8'))
        except (OSError, ValueError):
            print(f'[resume] {run.name}: unreadable manifest; ignoring')
            continue
        expected = {'model_id': model, 'n_prompts': n_prompts,
                    'max_new_tokens': tokens, 'seed': SEED}
        wrong = {k: (manifest.get(k), v) for k, v in expected.items()
                 if manifest.get(k) != v}
        if wrong:
            print(f'[resume] {run.name}: arguments differ {wrong}; ignoring')
            continue
        if list(manifest.get('schemes', [])) != schemes:
            print(f'[resume] {run.name}: schemes {manifest.get("schemes")} '
                  f'!= {schemes}; ignoring')
            continue
        digest = manifest.get('corpora', {}).get('prompts', {}).get('sha256_ordered')
        if corpus_sha and digest != corpus_sha:
            print(f'[resume] {run.name}: corpus hash differs; ignoring')
            continue
        ok = True
        for scheme in schemes:
            path = run / 'results' / f'completions_{scheme}.json'
            try:
                blob = json.loads(path.read_text(encoding='utf-8'))
                texts = blob['completions'] if isinstance(blob, dict) else blob
            except (OSError, ValueError, KeyError):
                print(f'[resume] {run.name}: {path.name} missing or unparseable')
                ok = False
                break
            if len(texts) != n_prompts:
                print(f'[resume] {run.name}: {path.name} has {len(texts)} rows, '
                      f'expected {n_prompts}')
                ok = False
                break
        if ok:
            complete.append(run)
    if len(complete) > 1:
        raise SystemExit(f'multiple completed runs share {label}: '
                         f'{[p.name for p in complete]}. Refusing to choose one.')
    return complete[0] if complete else None


def grade_complete(run, filename, schemes, n_prompts):
    '''A grading output that covers every scheme and every prompt.

    Same argument as above: a half-written grade file exists just as much as a
    finished one, and skipping the grader on the strength of that would leave
    the analysis reading rows that were never produced.
    '''
    path = run / 'results' / filename
    try:
        blob = json.loads(path.read_text(encoding='utf-8'))
    except (OSError, ValueError):
        return False
    # The scorer is part of what makes a grading the right grading. A file
    # written under first-token scoring satisfies every structural check here
    # while being the instrument this round exists to replace, so skipping on
    # its presence would silently keep the old measurement.
    if blob.get('scoring') not in (None, SCORING):
        print(f"[resume] {path.name}: scored under "
              f"{blob.get('scoring')!r}, not {SCORING!r}; will re-grade")
        return False
    results = blob.get('results') or blob.get('verdicts') or {}
    missing = [s for s in schemes if s not in results]
    if missing:
        print(f'[resume] {path.name}: no rows for {missing}; will re-grade')
        return False
    return True


def print_pairing(run):
    """Show the ordered corpus hash, so pairing is visible not assumed."""
    manifest = json.loads((run / 'manifest.json').read_text(encoding='utf-8'))
    digest = manifest.get('corpora', {}).get('prompts', {}).get('sha256_ordered')
    print(f'[run] {run}')
    print(f'[pairing] corpora.prompts.sha256_ordered = {digest}')


def record_skip(label, reason):
    RESULTS[label] = {'returncode': 0, 'skipped': True, 'minutes': 0.0,
                      'tail': [reason]}
    print(f'[{label}] already complete after Drive restore; skipping: {reason}')


## Step 1 — long-window HH-RLHF

The question: at 256 generated tokens, does the FP16-versus-4.5-bit behavioural comparison still hold for Qwen2.5-3B and Phi-3.5-mini? A result that survives here has survived a wider window, greedy decoding and one NF4 judge -- not a general test of the effect.

In [ ]:
for tag, model in MODELS_LONG:
    run_label, step_label = f'r3-long256-{tag}', f'long-{tag}'
    run_dir = completed_run(run_label, ['FP16', 'RTN_4B'], model,
                            2 * N_LONG, FOLD_A_SHA)
    if run_dir is None:
        print(f'\n{step_label}: will produce 500 HH-RLHF completions at 256 tokens for FP16 and RTN 4-bit; it tests whether the 4.5-bit result persists at a wider window for {model}.')
        free_vram()
        run_step_resumable(step_label, 'run_behavioural_ladder.py', ['--model', model, '--n', str(N_LONG), '--bits', '4', '--max-new-tokens', str(LONG_TOKENS), '--seed', str(SEED), '--no-activations', '--batch-size', '8', '--cache', BEHAV_CACHE, '--label', run_label], timeout=75 * 60)
        checkpoint_to_drive()
        run_dir = completed_run(run_label, ['FP16', 'RTN_4B'], model,
                            2 * N_LONG, FOLD_A_SHA)
    else:
        record_skip(step_label, str(run_dir))
    if run_dir is None:
        RESULTS[f'judge-long-{tag}'] = {'returncode': -1, 'blocked': True, 'minutes': 0.0, 'tail': ['generation failed']}
        print(f'[judge-long-{tag}] not run because its generation did not finish')
        checkpoint_to_drive()
        continue
    print_pairing(run_dir)
    judge_label = f'judge-long-{tag}'
    if grade_complete(run_dir, 'judge_classification.json',
                      ['FP16', 'RTN_4B'], 2 * N_LONG):
        record_skip(judge_label, str(run_dir / 'results' / 'judge_classification.json'))
    else:
        print(f'{judge_label}: will produce three-way 7B-NF4 judgements over saved 256-token completions; it grades that text through a 2000-character window, wide enough that a 256-token completion is read whole.')
        free_vram()
        # This grader has no --max-length argparse argument; passing one would make an unattended run fail.
        run_step_resumable(judge_label, 'classify_completions_judge.py', [str(run_dir), '--judge-model', JUDGE_MODEL, '--judge-4bit', '--completion-chars', str(JUDGE_COMPLETION_CHARS), '--scoring', SCORING, '--batch-size', '4'], timeout=30 * 60)
        checkpoint_to_drive()

    # The 48-token window onto the SAME generated text. Comparing this run
    # against the old 48-token run instead would assume batched greedy decoding
    # reproduced its own first 48 tokens at a different max_new_tokens -- a
    # floating-point determinism assumption, not a result. Deriving the prefix
    # from this run's own completions removes the assumption, and --compare
    # reports whether it would have held, which is worth knowing either way.
    prefix_label = f'r3-long256-{tag}-prefix48'
    prefix_dir = completed_run(prefix_label, ['FP16', 'RTN_4B'], model,
                               2 * N_LONG, FOLD_A_SHA, tokens=48)
    if prefix_dir is None:
        baseline = latest_run(f'*colab-behavioural-{tag}')
        cmd = [str(run_dir), '--tokens', '48']
        if baseline:
            cmd += ['--compare', str(baseline)]
        print(f'prefix-{tag}: deriving the 48-token prefix of this run')
        run_step(f'prefix-{tag}', 'make_prefix_run.py', cmd, timeout=10 * 60)
        checkpoint_to_drive()
        prefix_dir = completed_run(prefix_label, ['FP16', 'RTN_4B'], model,
                                   2 * N_LONG, FOLD_A_SHA, tokens=48)
    if prefix_dir is not None and not grade_complete(
            prefix_dir, 'judge_classification.json', ['FP16', 'RTN_4B'],
            2 * N_LONG):
        free_vram()
        run_step_resumable(
            f'judge-prefix-{tag}', 'classify_completions_judge.py',
            [str(prefix_dir), '--judge-model', JUDGE_MODEL, '--judge-4bit',
             '--completion-chars', str(JUDGE_COMPLETION_CHARS),
             '--scoring', SCORING, '--batch-size', '4'], timeout=30 * 60)
        checkpoint_to_drive()


## Step 2 — long-window XSTest baselines

This closes the labelled-arm gap: at 256 generated tokens, what are the FP16 baselines for harmful and benign XSTest prompts across the three models?

In [ ]:
for tag, model in MODELS_XSTEST:
    run_label, step_label = f'r3-xstest256-{tag}', f'xstest-{tag}'
    run_dir = completed_run(run_label, ['FP16'], model,
                            2 * N_XSTEST, XSTEST_SHA)
    if run_dir is None:
        print(f'\n{step_label}: will produce 300 labelled XSTest completions at 256 tokens for FP16; it supplies the labelled long-window baseline for {model}.')
        free_vram()
        run_step_resumable(step_label, 'run_behavioural_ladder.py', ['--model', model, '--prompts', 'data/eval_suites/xstest.jsonl', '--n', str(N_XSTEST), '--bits', '--max-new-tokens', str(LONG_TOKENS), '--seed', str(SEED), '--no-activations', '--batch-size', '8', '--cache', BEHAV_CACHE, '--label', run_label], timeout=25 * 60)
        checkpoint_to_drive()
        run_dir = completed_run(run_label, ['FP16'], model,
                            2 * N_XSTEST, XSTEST_SHA)
    else:
        record_skip(step_label, str(run_dir))
    if run_dir is None:
        RESULTS[f'taxonomy-xstest-{tag}'] = {'returncode': -1, 'blocked': True, 'minutes': 0.0, 'tail': ['generation failed']}
        print(f'[taxonomy-xstest-{tag}] not run because its generation did not finish')
        checkpoint_to_drive()
        continue
    print_pairing(run_dir)
    grade_label = f'taxonomy-xstest-{tag}'
    if grade_complete(run_dir, 'completion_taxonomy.json',
                      ['FP16'], 2 * N_XSTEST):
        record_skip(grade_label, str(run_dir / 'results' / 'completion_taxonomy.json'))
    else:
        print(f'{grade_label}: will produce five-way 7B-NF4 taxonomy judgements over labelled 256-token completions; it separates refusal, deflection, disclaimer, compliance and unclear responses.')
        free_vram()
        run_step_resumable(grade_label, 'classify_completion_taxonomy.py', [str(run_dir), '--judge-model', JUDGE_MODEL, '--judge-4bit', '--completion-chars', str(JUDGE_COMPLETION_CHARS), '--max-length', str(TAXONOMY_MAX_LENGTH), '--scoring', SCORING, '--batch-size', '4'], timeout=18 * 60)
        checkpoint_to_drive()


## Step 3 — re-grade the published 48-token runs with the corrected scorer

The cheapest measurement in this notebook, and the only one that speaks
directly to the numbers already in the paper.

The published verdicts were read off first-token logits, comparing four
three-character label prefixes against one whole word. Letter scoring replaces
that with five verified single-token options. Running it over the *existing*
48-token completions -- text that is already stored, so nothing is generated --
measures exactly how much the published numbers move when only the scorer is
corrected. Same completions, same judge, same window; one thing changed.

If the answer is "barely", the first-token caveat can be retired. If the
harmful-compliance cell stops being empty, that cell was a label-token artifact
and not only a window artifact, which no amount of 256-token generation would
have revealed.

Skipped without complaint if the old run directories are not in Drive. It is
reported rather than assumed, because a step that silently does nothing looks
identical to one that ran and found no change.

In [ ]:
# Only FP16 and the 4.5-bit rung: those carry every published comparison, and
# grading the whole ladder again would cost more than the rest of this notebook.
REGRADE = [('qwen3b', '*colab-behavioural-qwen3b', ['FP16', 'RTN_4B'], 500,
            'classify_completions_judge.py'),
           ('phi35', '*colab-behavioural-phi35', ['FP16', 'RTN_4B'], 500,
            'classify_completions_judge.py'),
           ('qwen3b-xstest', '*lab-qwen3b-xstest', ['FP16'], 300,
            'classify_completion_taxonomy.py'),
           ('phi35-xstest', '*lab-phi35-xstest', ['FP16'], 300,
            'classify_completion_taxonomy.py'),
           ('smol17-xstest', '*lab-smol17-xstest', ['FP16'], 300,
            'classify_completion_taxonomy.py')]

missing = []
for tag, pattern, schemes, n_prompts, script in REGRADE:
    run_dir = latest_run(pattern)
    if run_dir is None:
        missing.append(tag)
        continue
    label = f'regrade-{tag}'
    free_vram()
    args = [str(run_dir), '--judge-model', JUDGE_MODEL, '--judge-4bit',
            '--completion-chars', str(JUDGE_COMPLETION_CHARS),
            '--batch-size', '4', '--schemes', *schemes]
    if script == 'classify_completion_taxonomy.py':
        args += ['--max-length', str(TAXONOMY_MAX_LENGTH)]
    # BOTH modes, explicitly. Letter scoring alone would leave the comparison
    # resting on whatever first-token caches happened to survive in Drive --
    # which is not something to discover afterwards. Running both here makes
    # the disagreement a measurement on identical text rather than a hope, and
    # the two write separate cache files because the mode is in the
    # fingerprint. First-token runs second so the letter verdicts, the ones we
    # intend to use, land first if the session is cut short.
    for mode in ('letter', 'first-token'):
        print(f'{label} [{mode}]: re-grading {run_dir.name} on {schemes}')
        run_step_resumable(f'{label}-{mode}', script,
                           args + ['--scoring', mode], timeout=25 * 60)
        checkpoint_to_drive()

if missing:
    print(f'NOT RE-GRADED (no run directory restored from Drive): {missing}')
    print('These are the published 48-token runs. Without them the letter-'
          'versus-first-token comparison cannot be made and the first-token '
          'caveat stays in the paper. Upload artifacts/runs to Drive and rerun '
          'this cell if you want that comparison.')
else:
    print('all five published runs re-graded under the corrected scorer')

## Export

The archive contains only round-three run directories, with paths relative to the repository root. A failed or blocked step makes the archive name begin with INCOMPLETE and prints the scientific question that remains unanswered.

In [ ]:
import zipfile
# The r3-* runs, PLUS the published runs re-graded in step 3. Those are old
# directories with new grade files inside them, so an r3-only glob would leave
# the letter-versus-first-token comparison on the Colab disk and ship an
# archive that cannot answer the cheapest question in the notebook.
runs = sorted(pathlib.Path('artifacts/runs').glob('*_r3-*'))
regraded = [latest_run(p) for _t, p, *_r in REGRADE]
runs += [r for r in regraded if r is not None and r not in runs]
runs = sorted(set(runs))
print('round-three run directories:')
for run in runs:
    print('  ', run.name)

steps = {key: value for key, value in RESULTS.items() if '-retry' not in key}
failed = [key for key, value in steps.items() if value.get('returncode') != 0]
# Absence is not success. If step 3 found no run directory to re-grade, nothing
# was recorded in RESULTS at all, and an archive whose only signal is 'no
# failures' would read as complete while the comparison it was built for is
# missing.
for _tag, _pattern, *_rest in REGRADE:
    if not any(k.startswith(f'regrade-{_tag}') for k in RESULTS):
        failed.append(f'regrade-{_tag} (never ran)')
retried = sorted({key.split('-retry')[0] for key in RESULTS if '-retry' in key})
status = {'steps': RESULTS, 'runs': [run.name for run in runs], 'failed': failed, 'retried': retried, 'long_models': [tag for tag, _ in MODELS_LONG], 'xstest_models': [tag for tag, _ in MODELS_XSTEST], 'n_long_per_class': N_LONG, 'n_xstest_per_class': N_XSTEST, 'max_new_tokens': LONG_TOKENS, 'completion_chars': JUDGE_COMPLETION_CHARS, 'taxonomy_max_length': TAXONOMY_MAX_LENGTH}
status_path = pathlib.Path('artifacts/runs/ROUND3_STATUS.json')
status_path.write_text(json.dumps(status, indent=2), encoding='utf-8')
checkpoint_to_drive()

stamp = time.strftime('%Y%m%d-%H%M%S')
prefix = 'INCOMPLETE_' if failed else ''
archive = pathlib.Path((f'/content/{prefix}cliffguard_round3_{stamp}.zip' if IN_COLAB else f'{prefix}cliffguard_round3_{stamp}.zip'))
# The status JSON goes IN the archive, not just beside it. The archive is what
# leaves this machine; a reader who has only the zip must be able to tell which
# steps ran, which failed, and which were restored from a previous session --
# otherwise a step that never ran is indistinguishable from one that ran and
# found nothing, which is the single most dangerous confusion this project has.
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(status_path, status_path.as_posix())
    for run in runs:
        for path in sorted(run.rglob('*')):
            if path.is_file():
                zf.write(path, path.as_posix())
print(f'\nwrote {archive}  ({archive.stat().st_size / 1e6:.1f} MB)')
print(f'wrote status JSON: {status_path}')
print('\nSTEPS:')
for key, value in steps.items():
    mark = 'ok' if value.get('returncode') == 0 else f'FAIL({value.get("returncode")})'
    suffix = ' (restored)' if value.get('skipped') else ''
    print(f'  {mark:9s} {key:28s} {value.get("minutes", 0):6.1f} min{suffix}')
if failed:
    print('\nINCOMPLETE RUN — do not report a missing comparison as a null result.')
    if any(key.startswith('long-') or key.startswith('judge-long-') for key in failed):
        print('UNANSWERED: whether the 256-token HH-RLHF FP16-versus-4-bit headline holds for every requested model.')
    if any(key.startswith('xstest-') or key.startswith('taxonomy-xstest-') for key in failed):
        print('UNANSWERED: the labelled 256-token XSTest FP16 baseline and its refusal/deflection taxonomy for every requested model.')
else:
    print('\nAll requested round-three measurements and grades completed.')
if retried:
    print('resumed after an OOM:', retried)
if IN_COLAB:
    try:
        from google.colab import files
        files.download(str(archive))
    except Exception as exc:
        print('download it from the file browser instead:', exc)
